# Pramana: fine-tune a small grounded field extractor

QLoRA on Qwen3-1.7B (Apache-2.0) with Unsloth, trained on synthetic AICTE document prompts from `training/make_dataset.py`. Exports a Q4_K_M GGUF for Ollama.

1. Runtime > Change runtime type > T4 GPU (free tier).
2. Run all. Upload `train.jsonl` and `val.jsonl` from `backend/training/data/` when asked.
3. Download `pramana-extract-q4_k_m.gguf` at the end, then on the laptop: `ollama create pramana-extract -f backend/training/Modelfile`.

Training data is synthetic. The model only proposes values; Pramana accepts a value only if it is found on the page.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade --no-deps "trl>=0.19" datasets

In [ ]:
import os
if not (os.path.exists('train.jsonl') and os.path.exists('val.jsonl')):
    from google.colab import files
    files.upload()  # select train.jsonl and val.jsonl
print(os.path.getsize('train.jsonl'), os.path.getsize('val.jsonl'))

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

BASE = 'unsloth/Qwen3-1.7B'
MAX_LEN = 2048
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=26261)

ds = load_dataset('json', data_files={'train': 'train.jsonl', 'validation': 'val.jsonl'})
def to_text(ex):
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False, enable_thinking=False)}
ds = ds.map(to_text, remove_columns=ds['train'].column_names)
print(ds['train'][0]['text'][:600])

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds['train'], eval_dataset=ds['validation'],
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, per_device_train_batch_size=4,
                   gradient_accumulation_steps=4, num_train_epochs=2, learning_rate=2e-4, warmup_ratio=0.05,
                   lr_scheduler_type='cosine', logging_steps=10, eval_strategy='epoch', optim='adamw_8bit',
                   weight_decay=0.01, seed=26261, output_dir='out', report_to='none'))
# Learn only the JSON answers, not the prompts
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n',
                                  response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)
print(trainer.evaluate())

In [ ]:
# Quick sanity check on one validation prompt
import json
FastLanguageModel.for_inference(model)
row = json.loads(open('val.jsonl', encoding='utf-8').readline())
prompt = tok.apply_chat_template(row['messages'][:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
ids = tok(prompt, return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
print('expected:', row['messages'][2]['content'])

In [ ]:
model.save_pretrained_gguf('pramana-extract', tok, quantization_method='q4_k_m')
import glob, shutil
g = sorted(glob.glob('pramana-extract*/**/*.gguf', recursive=True) + glob.glob('pramana-extract*.gguf'))
print(g)
shutil.copy(g[-1], 'pramana-extract-q4_k_m.gguf')
from google.colab import files
files.download('pramana-extract-q4_k_m.gguf')